# 02 — Gauge Preprocessing + Completeness QC (FIXED)

Cleans monthly BMD gauge data and enforces one record per
`station_id + year + month`. Missing months, coordinate inconsistency and extreme values are flagged.

Zero rainfall is preserved as valid data. Suspicious high values are flagged, not automatically deleted.

In [1]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import re
import numpy as np
import pandas as pd

gauge_files = sorted((RAW_DIR / "gauge").glob("*.csv"))
if not gauge_files:
    raise FileNotFoundError("No gauge CSV found in data/raw/gauge")

# Prefer the known file if present.
known = RAW_DIR / "gauge" / "bmd_monthly_rainfall_2017_2022.csv"
gauge_path = known if known.exists() else gauge_files[0]
print("Gauge source:", gauge_path)

df = pd.read_csv(gauge_path)
print("Raw shape:", df.shape)
print("Raw columns:", list(df.columns))

Gauge source: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\gauge\bmd_monthly_rainfall_2017_2022.csv
Raw shape: (432, 8)
Raw columns: ['station_id', 'station_name', 'latitude', 'longitude', 'year', 'month', 'date', 'rainfall_mm']


In [3]:
def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

df.columns = [norm_col(c) for c in df.columns]

aliases = {
    "station": "station_id",
    "station_name": "station_id",
    "stationid": "station_id",
    "rainfall": "rainfall_mm",
    "precipitation": "rainfall_mm",
    "rainfall_mm_month": "rainfall_mm",
    "lat": "latitude",
    "lon": "longitude",
    "lng": "longitude",
}

for old, new in aliases.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old:new})

required = ["station_id", "year", "month", "rainfall_mm", "latitude", "longitude"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required gauge columns: {missing}. Available: {list(df.columns)}")

for c in ["year","month","rainfall_mm","latitude","longitude"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["station_id"] = df["station_id"].astype(str).str.strip()
df = df[df["station_id"].ne("") & df["station_id"].ne("nan")].copy()
df = df[df["year"].between(2017, 2022)]
df = df[df["month"].between(1,12)]
df = df[df["rainfall_mm"].ge(0) | df["rainfall_mm"].isna()]

# Physically valid coordinate ranges only.
df.loc[~df["latitude"].between(-90,90), "latitude"] = np.nan
df.loc[~df["longitude"].between(-180,180), "longitude"] = np.nan

df["date"] = pd.to_datetime(dict(year=df.year.astype("Int64"),
                                  month=df.month.astype("Int64"),
                                  day=1), errors="coerce")

In [4]:
# Enforce uniqueness of station-year-month.
key = ["station_id","year","month"]
dup = df[df.duplicated(key, keep=False)].sort_values(key)
if len(dup):
    print("ERROR: conflicting/duplicate station-month records found:")
    display(dup)
    # Exact duplicate rows may be safely removed, but conflicting values must stop the workflow.
    exact_dedup = df.drop_duplicates()
    conflicting = exact_dedup[exact_dedup.duplicated(key, keep=False)]
    if len(conflicting):
        raise ValueError("Conflicting duplicate station-year-month records must be resolved in the raw gauge CSV.")
    df = exact_dedup.copy()

# Coordinates should remain constant for a station.
coord_qc = df.groupby("station_id").agg(
    n_lat=("latitude","nunique"),
    n_lon=("longitude","nunique"),
    latitude=("latitude","median"),
    longitude=("longitude","median"),
).reset_index()

display(coord_qc)
if ((coord_qc.n_lat > 1) | (coord_qc.n_lon > 1)).any():
    print("WARNING: Some station coordinates change through time. Review before modelling.")

# Completeness table.
completeness = df.groupby(["station_id","year"]).size().rename("months").reset_index()
display(completeness)

expected = pd.MultiIndex.from_product(
    [sorted(df.station_id.unique()), range(2017,2023), range(1,13)],
    names=["station_id","year","month"]
)
actual = pd.MultiIndex.from_frame(df[key])
missing_idx = expected.difference(actual)
print("Missing station-month records:", len(missing_idx))
if len(missing_idx):
    display(missing_idx.to_frame(index=False))

,station_id,n_lat,n_lon,latitude,longitude
0,CL503,1,1,22.6012,89.5195
1,CL504,1,1,22.8093,89.4145
2,CL509,1,1,22.6887,89.3088
3,CL510,1,1,22.8319,89.5500
4,CL515,1,1,22.5850,89.3182
5,CL517,1,1,22.7900,89.5900


,station_id,year,months
0,CL503,2017,12
1,CL503,2018,12
2,CL503,2019,12
3,CL503,2020,12
4,CL503,2021,12
5,CL503,2022,12
6,CL504,2017,12
7,CL504,2018,12
8,CL504,2019,12
9,CL504,2020,12


Missing station-month records: 0


In [5]:
# Rainfall QC: flag, do not automatically remove plausible monsoon extremes.
desc = df["rainfall_mm"].describe(percentiles=[.01,.05,.50,.95,.99])
display(desc.to_frame("rainfall_mm"))

q1, q3 = df["rainfall_mm"].quantile([.25,.75])
iqr = q3 - q1
high_flag = q3 + 3*iqr
df["qc_extreme_high"] = df["rainfall_mm"] > high_flag
print("Extreme-high review threshold:", high_flag)
print("Flagged rows:", int(df["qc_extreme_high"].sum()))
if df["qc_extreme_high"].any():
    display(df.loc[df.qc_extreme_high, ["station_id","year","month","rainfall_mm"]])

# Rows missing essential fields cannot be used.
essential = ["station_id","year","month","rainfall_mm","latitude","longitude"]
before = len(df)
df = df.dropna(subset=essential).copy()
print("Dropped rows with missing essential fields:", before-len(df))

df = df.sort_values(["station_id","year","month"]).reset_index(drop=True)
out = PROCESSED_DIR / "gauge_monthly_clean.csv"
df.to_csv(out, index=False)
completeness.to_csv(PROCESSED_DIR / "gauge_completeness_qc.csv", index=False)
coord_qc.to_csv(PROCESSED_DIR / "gauge_coordinate_qc.csv", index=False)
print("Saved:", out, df.shape)

,rainfall_mm
count,432.000000
mean,158.717361
std,159.766900
min,0.000000
1%,0.000000
5%,0.000000
50%,109.450000
95%,460.545000
99%,671.500000
max,783.000000


Extreme-high review threshold: 860.1999999999999
Flagged rows: 0
Dropped rows with missing essential fields: 0
Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\gauge_monthly_clean.csv (432, 9)
